**Policy RAG | Step 1: Policy chunking**

Goal: split the marketplace policy into one chunk per clause, each with an ID that can be cited in an answer.
Reusable code lives in `src/policy_rag.py`. This notebook calls it and checks the result.

**Step. Load the policy.** Reads the policy file and keeps non-empty lines only.

In [1]:
import sys
import pandas as pd

sys.path.append("../../src")
from policy_rag import load_policy, chunk_policy

lines = load_policy("policy/marketplace_policy.txt")
for ln in lines[:4]:
    print("-", ln[:90])

- Appendix A — Marketplace Operations Policy
- Fictional, written for this exercise. Save it into your repo as your RAG corpus. Splitting
- MOP-1. Seller Performance Tiers
- 1.1 Every active seller is assigned a performance tier each calendar month, calculated on 


The first two are the title and a note, not policy text, so they are ignored. Section headers look like `MOP-1.` and clauses look like `1.1`.

**Step. Chunk by clause.** A new chunk starts at each clause number. Table rows, bullets and numbered items stay inside the clause above them.

**The idea**

The policy file is a list of lines. The code reads the lines from top to bottom and asks one question for each line: does this line start a new chunk, or does it belong to the chunk above?

The rule:
- If a line starts with a clause number (like `1.1`, `4.1`, `6.4`), a new chunk starts.
- If a line does not start with a clause number, it is a continuation. It is added to the chunk above.

**Example with clause 4.1 (bullets)**

These are the lines in the file:

```
4.1 A cash refund to the original payment method may be issued where any one of the following holds:
- The order is classified Critical under MOP-2
- The order was cancelled before carrier handoff
- The customer has an open consumer protection complaint registered with Procon
- The customer account has been closed and platform credit cannot be applied
4.2 Cash refunds are processed within 10 business days for credit card payments...
```

What the code does, line by line:
- `4.1 A cash refund...` starts with a clause number, so a new chunk `MOP-4.1` starts.
- The four lines starting with `•` have no clause number, so each is added to `MOP-4.1`.
- `4.2 Cash refunds...` starts with a clause number, so a new chunk `MOP-4.2` starts.

Result: chunk `MOP-4.1` holds the sentence and all four conditions together.

**Example with a numbered list (clause 6.4)**

```
6.4 Before any suspension, the Quality Team must complete and record all of the following:
1. Written notice to the seller...
2. A review of open Liability Waiver Requests...
3. Confirmation that outstanding customer orders...
4. Sign-off from a Quality Team lead
6.5 Suspended sellers may apply...
```

The items `1.`, `2.`, `3.`, `4.` are not treated as clauses. A clause number has a digit, a dot and another digit (`6.4`). A list item has a digit and a dot, then a space (`1. Written`). So the four items stay inside `MOP-6.4`.

**Example with a one-line clause (clause 1.3)**

```
1.3 A seller must satisfy both thresholds for a tier. A seller with 96% OTD and 11% ARR is Bronze, not Gold.
1.4 Tier changes take effect on the first business day of the month...
```

Here nothing needs to be added. `1.3` starts a chunk, and the next line `1.4` starts a new one, so `MOP-1.3` is just this one line.

**Why we do it this way**

In clause 4.1, the sentence says "any one of the following holds", and the four bullets are the real answer. If each bullet were its own chunk, a question like "When can a customer get a cash refund?" could find the sentence without the conditions, or one condition without the others. The answer would be incomplete.

Clause 6.4 is the same. The sentence says "must complete all of the following", and the four items are the actual steps.

**Short version**

One clause is one chunk. Everything under a clause number belongs to that clause until the next clause number appears.

In [2]:
chunks = chunk_policy(lines)
df = pd.DataFrame(chunks)
df["words"] = df["text"].str.split().str.len()

print("total chunks:", len(df))
df.groupby(["section_no", "section_title"]).size().rename("clauses").to_frame()

total chunks: 33


,,clauses
section_no,section_title,
1,Seller Performance Tiers,5
2,Delivery Delay Classification,4
3,Customer Compensation,5
4,Cash Refund Eligibility,3
5,Review Moderation,4
6,Seller Escalation and Suspension,5
7,Payouts,3
8,Freight Cost Allocation,4


The policy gives 33 chunks across 8 sections. Sections hold between 3 and 5 clauses each.
- total chunks: 33
- The code cut the policy into 33 pieces. Each piece is one clause (1.1, 1.2, 2.1 and so on).

**The small table below it:** It shows how many clauses each section has. The columns are:
- section_no: the section number (1 to 8, for MOP-1 to MOP-8)
- section_title: the name of that section
- clauses: how many chunks (clauses) belong to that section

**Reading the rows:**
- Section 1, Seller Performance Tiers, has 5 clauses (1.1 to 1.5).
- Section 4, Cash Refund Eligibility, has 3 clauses (4.1 to 4.3).
- Section 6, Seller Escalation and Suspension, has 5 clauses (6.1 to 6.5).

**Check the total**: 5 + 4 + 5 + 3 + 4 + 5 + 3 + 4 = 33. The section counts add up to the total, so no clause was lost or counted twice.

**Step. Chunk size.** Word counts per chunk, and the chunks that span more than one line.

In [3]:
print(df["words"].describe().round(1).to_string())
print()
print("chunks with a table or list inside:")
df[df["text"].str.contains("\n")][["chunk_id", "words"]]

count    33.0
mean     34.1
std      14.3
min      18.0
25%      22.0
50%      29.0
75%      43.0
max      67.0

chunks with a table or list inside:


,chunk_id,words
1,MOP-1.2,60
6,MOP-2.2,59
10,MOP-3.2,47
14,MOP-4.1,60
24,MOP-6.4,67


Chunks are short: the smallest has 18 words, the median is 29 and the largest is 67. A clause is already a small unit, so no further splitting is needed. Five clauses (`MOP-1.2`, `MOP-2.2`, `MOP-3.2`, `MOP-4.1`, `MOP-6.4`) hold a table or list and stay whole, so a rule is never separated from its table.

**Step. Cross-references.** Finds clauses that mention another clause or section, and checks each target exists.

In [4]:
valid_ids = set(df["chunk_id"]) | {f"MOP-{s}" for s in df["section_no"]}

refs = df[df["refs"].str.len() > 0][["chunk_id", "refs"]].reset_index(drop=True)

print("clauses with a reference:", len(refs))
refs

clauses with a reference: 10


,chunk_id,refs
0,MOP-1.1,[MOP-6]
1,MOP-1.5,[MOP-7]
2,MOP-2.3,[MOP-2.2]
3,MOP-3.1,[MOP-4]
4,MOP-3.4,[MOP-3.2]
5,MOP-4.1,[MOP-2]
6,MOP-5.2,[MOP-2.3]
7,MOP-6.4,[MOP-6.3]
8,MOP-7.2,[MOP-2]
9,MOP-8.4,"[MOP-2.2, MOP-2.4]"


10 of 33 clauses refer to another clause or section. Some answers will need two clauses together, for example `MOP-8.4` points to both `MOP-2.2` and `MOP-2.4`. Whether to pull the referenced clause in automatically is decided, after seeing which questions fail.

**Step. Preview.** Two chunks printed in full, one with a table and one with a list.

In [5]:
for cid in ["MOP-1.2", "MOP-6.4"]:
    row = df[df["chunk_id"] == cid].iloc[0]
    print(f"[{row.chunk_id}] {row.section_title}")
    print(row.text)
    print()

[MOP-1.2] Seller Performance Tiers
1.2 Tiers are determined by two measures: on-time delivery rate (OTD) and adverse review rate (ARR, the share of orders receiving 1 or 2 stars).
Tier | OTD | ARR
Gold | ≥ 95% | ≤ 4%
Silver | ≥ 88% | ≤ 9%
Bronze | ≥ 75% | ≤ 15%
Watch | below Bronze on either measure | —

[MOP-6.4] Seller Escalation and Suspension
6.4 Before any suspension, the Quality Team must complete and record all of the following:
1. Written notice to the seller with at least 10 business days to respond, except where 6.3 immediate grounds apply
2. A review of open Liability Waiver Requests, which must be resolved first
3. Confirmation that outstanding customer orders have a fulfilment or refund plan
4. Sign-off from a Quality Team lead



Tables and lists are kept as written, with the clause number at the top. This text is what will be embedded and shown to the LLM as evidence.

**Decisions from this step**
- One chunk per clause, because clauses are short and self-contained.
- The citation ID is `MOP-<section>.<clause>`, for example `MOP-6.4`.
- Section title is kept as metadata. Next Step will test adding it to the text that is embedded.